In [1]:
# Load in Data and Libraries
import pandas as pd
import numpy as np
from pathlib import Path

# The notebook lives in notebooks/, so the project root is one folder up
PROJECT_ROOT  = Path.cwd().parent
RAW_DATA      = PROJECT_ROOT / "data" / "raw"

tix = pd.read_csv(RAW_DATA / "ticket_activity_2025_26.csv", dtype=str, keep_default_na=False)
ren = pd.read_csv(RAW_DATA / "renewals_2026_27.csv", dtype=str, keep_default_na=False)

In [2]:
# Noticed Uppercase/Lowercase. Will need to standardize the case in columns for consistency.
tix = tix.apply(lambda col: col.str.strip().str.lower() if col.dtypes == 'object' else col)
ren = ren.apply(lambda col: col.str.strip().str.lower() if col.dtypes == 'object' else col)

In [3]:
# Question Trying to Answer: How many rows and columns are there, what what does one row represent?

# Read First 10 Rows of Each Data Set
display(tix.head(10))
display(ren.head(10))

# How many rows and columns are within each dataset?
print("Ticket Activity Data Set:") 
print(tix.shape) #(25000, 25)
print("Renewals Data Set:")
print(ren.shape) # (867, 4)

,Account_ID,Name,Email,Age,Location,Account_Type,Member_Since,Event_Name,Event_Type,Event_Date,...,Ticket_Type,Price_Per_Seat,Payment_Plan,Payment_Status,Seat_Status,Tickets_Scanned,Resale_Price,Rep_Code,Notes,Unnamed: 23
0,ACCT100000,Danielle Johnson,danielle.johnson@gmail.com,31,"New York, NY",Individual,2024,Knicks Home Game 01 vs Magic,Game,2025-11-01,...,Full Season,$405.00,Pay in Full,GREEN,,3,,jm,,
1,ACCT100000,Danielle Johnson,danielle.johnson@gmail.com,31,"New York, NY",Individual,2024,Knicks Home Game 02 vs Bulls,Game,2025-11-02,...,Full Season,405.0,Pay in Full,Green,,3,,,,
2,ACCT100000,Danielle Johnson,danielle.johnson@gmail.com,31,"New York, NY",Individual,2024,Knicks Home Game 03 vs Mavericks,Game,11/04/2025,...,Full Season,405.0,Pay in Full,Green,,3,,,,
3,ACCT100000,Danielle Johnson,danielle.johnson@gmail.com,31,"New York, NY",Individual,2024,Knicks Home Game 04 vs Jazz,Game,2025-11-06,...,Full Season,$405.00,Pay in Full,Green,,4,,jm,,
4,ACCT100000,Danielle Johnson,danielle.johnson@gmail.com,31,"New York, NY",Individual,2024,Knicks Home Game 05 vs Clippers,Game,11/10/2025,...,Full Season,405.0,,Green,,,,,,
5,ACCT100000,Danielle Johnson,danielle.johnson@gmail.com,31,"New York, NY",Individual,2024,Knicks Home Game 06 vs 76ers,Game,2025-11-19,...,Full Season,405.0,Pay in Full,Green,,4,,,,
6,ACCT100000,Danielle Johnson,danielle.johnson@gmail.com,31,"New York, NY",Individual,2024,Knicks Home Game 07 vs Hawks,Game,11/28/2025,...,Full Season,405.0,PIF,Green,,2,,,,
7,ACCT100000,Danielle Johnson,danielle.johnson@gmail.com,31,"New York, NY",Individual,2024,NYK HG08 vs Suns,Game,2025-12-08,...,Full Season,$405.00,Pay in Full,Green,,3,,KT,,
8,ACCT100000,Danielle Johnson,danielle.johnson@gmail.com,31,"New York, NY",Individual,2024,Knicks Home Game 09 vs Mavericks,Game,12/18/2025,...,Full Season,$405.00,Pay in Full,Green,,4,,,,
9,ACCT100000,Danielle Johnson,danielle.johnson@gmail.com,31,"New York, NY",Individual,2024,Knicks Home Game 10 vs Magic,Game,2025-12-20,...,,405.0,Pay in Full,Green,,3,,,,


,Account_ID,Renewal_Status,Cancellation_Reason,Processed_Date
0,ACCT100668,Renewed,,05/19/2026
1,ACCT100030,N,Price increase,05/29/2026
2,ACCT100377,Cancelled,Bought elsewhere,05/22/2026
3,ACCT100535,Renewed,,06/13/2026
4,ACCT100806,Renewed,,05/24/2026
5,ACCT100792,Renewed,,05/31/2026
6,ACCT100363,Renewed,,06/04/2026
7,ACCT100583,Renewed,,06/08/2026
8,ACCT100165,Renewed,,05/03/2026
9,ACCT100483,Canceled,Team performance,05/17/2026


Ticket Activity Data Set:
(25000, 25)
Renewals Data Set:
(889, 4)


# Missing Values within Tickets and Renewal Dataset. 
## Shows us:
- Number of Values within Columns missing missing
- Can values be replicated from other column valuess
- How important it is for model

In [4]:
# Missing Values per Column in the Ticket Activity Data Set  

blank_words = ["", "na", "n/a", "null", "none", "-", "unknown", "nan"]
is_blank = tix.isin(blank_words)

blank_summary = pd.DataFrame({
    "blank_count": is_blank.sum(),
    "blank_pct": (is_blank.mean()*100).round(2)
}).sort_values("blank_count", ascending = False)

blank_summary

,blank_count,blank_pct
Unnamed: 23,25000,100.00
Resale_Price,24495,97.98
Notes,24492,97.97
Seat_Status,20317,81.27
Rep_Code,18660,74.64
Location,1792,7.17
Account_Type,1490,5.96
Age,1433,5.73
Ticket_Type,1280,5.12
Price_Per_Seat,1270,5.08


# Summary of Blanks: Ticket Activity Dataset (25,000 event rows)
### 6 Sections: Junk, Not Needed for Model, Insightful, Missing (Fillable), Truly Missing, Complete

### Junk
- Unnamed: 23 (100% blank, export artifact)
- Notes (98% blank free-text field; could become a "has a note" flag, but too sparse to help the model)

### Not Needed for Model
- Email (4.7% blank): contact info, doesn't predict renewal
- Name (1.9% blank): contact info, doesn't predict renewal

### Insightful (blank carries meaning)
- Resale Price (98% blank): blank means the seat was not resold for that event
- Seat Status (81% blank): blank means the holder kept the seat for that event
- Rep Code (75% of event rows blank): blank likely means no rep assigned. Does having a rep relate to renewal?

### Missing (Fillable)
- Location (7.2%): same Account ID's other rows
- Account Type (6.0%): same Account ID. Does account type affect renewal?
- Age (5.7%): same Account ID
- Ticket Type (5.1%): from the Price Code suffix (_F, _H, _Q)
- Price Per Seat (5.1%): same Account ID or same Price Code
- Member Since (4.1%): same Account ID
- Payment Plan (3.1%): same Account ID
- Event Type (3.0%): from Event Name ("Home Game" vs. "Concert")
- Payment Status (1.8%): same Account ID (account-level billing status)
- Event Date (1.0%): other rows with the same Event Name
- Number of Seats (1.0%): count from Seat Numbers ("8-11" = 4) or the account's other rows

*To test:* check whether any accounts are blank on every row for these columns. If so, those values can't be filled and move to Truly Missing.

### Truly Missing
- Tickets Scanned (2.0%): event-specific, so other rows can't fill it. Leave as NaN and compute the no-show rate only from rows where it's known.

### Complete (no blanks)
- Account ID, Event Name, Price Code, Section, Row, Seat Numbers

In [5]:
# Missing Values per Column in the Renewals Data Set
blank_words = ["", "na", "n/a", "null", "none", "-", "unknown", "nan"]
is_blank_ren = ren.isin(blank_words)

blank_summary_ren = pd.DataFrame({
    "blank_count": is_blank_ren.sum(),
    "blank_pct": (is_blank_ren.mean()*100).round(2)
}).sort_values("blank_count", ascending = False)

blank_summary_ren

,blank_count,blank_pct
Cancellation_Reason,750,84.36
Account_ID,0,0.00
Renewal_Status,0,0.00
Processed_Date,0,0.00


# Summary of Blanks: Renewal Dataset (889 accounts, one row per account)

## Complete
- Account_ID, Renewal_Status, Processed_Date: no blanks

## Insightful
- Cancellation_Reason: 750 blanks (84%). Blank mostly means the member renewed
  (737 renewers + 13 cancellers who gave no reason).
- Overall: 152 of 889 accounts (17.1%) cancelled, so about 83% renewed.
  The classes are imbalanced, so accuracy alone will be misleading.

## Not usable as features (leakage)
- Cancellation_Reason and Processed_Date are only known after the renewal decision.

# Value Counts within Each Dataset
## Count how many Rows are Assigned to each value. Hels us better understand the data and how it is distributed, as well as how messy it is. 

In [6]:
# Value Counts for Categorical Columns in the Ticket Activity Datatset:
for col in ["Payment_Plan", "Payment_Status", "Seat_Status", "Account_Type", "Event_Type", "Ticket_Type"]:
      print(tix[col].str.strip().value_counts(), "\n")

Payment_Plan
Pay in Full               10240
12 Payments, Monthly       4078
3 Way Payment Plan         3287
5-6 Month Payment Plan     3176
                            769
PIF                         624
Pay In Full                 619
pay in full                 588
Monthly (12)                311
12 Monthly                  301
3 way pmt plan              261
6 Month Plan                257
3-Pay                       247
5-6 Mo Plan                 242
Name: count, dtype: int64 

Payment_Status
Green     18619
Yellow     2133
Red        1041
Grn         816
GREEN       780
green       771
            460
Yel         113
YELLOW       94
yellow       83
red          52
RED          38
Name: count, dtype: int64 

Seat_Status
       20347
F       1972
N/A      573
-        573
RE       493
O        372
D        308
P        255
f         60
d         14
o         13
re        12
p          8
Name: count, dtype: int64 

Account_Type
Individual    18627
Corporate      4211
              

In [7]:
# Value Counts in the Categorical Columns within the Renewals Data Set:
for col in ["Renewal_Status", "Cancellation_Reason"]:
    print(ren[col].str.strip().value_counts(),"\n")

Renewal_Status
Renewed      526
Cancelled     88
Y             85
RENEWED       66
Yes           34
Canceled      31
renewed       26
CANCELLED     14
N             10
No             9
Name: count, dtype: int64 

Cancellation_Reason
                        750
Price increase           25
Bought elsewhere         25
Team performance         25
Moving away              22
Not attending enough     20
Financial                12
Company budget cut       10
Name: count, dtype: int64 



# Value Counts Summary Across Datasets

## Ticket Activity Dataset (25,000 event rows, 917 accounts)

### Messy values that need consolidating
| Column | Raw spellings | After strip + lowercase | Should be |
|---|---|---|---|
| Payment_Plan | 14 | 12 | 4 (Pay in Full, 3 Way, 5-6 Month, 12 Monthly) |
| Payment_Status | 13 | 6 | 3 (Green, Yellow, Red) |
| Seat_Status | 14 | 8 | 6 (RE, P, F, O, D, Kept) |
| Account_Type | 5 | 3 | 2 (Individual, Corporate) |
| Event_Type | 5 | 3 | 2 (Game, Concert) |

- Stripping whitespace and lowercasing fixes most of the variants. The rest
  (e.g. "PIF", "Grn", "N/A") need a mapping dictionary.

### Early patterns
- **Payment plan:** about 48% of rows, and about 49% of accounts, are Pay in Full.
  The other half are split across the three installment plans.
- **Account type:** 793 of 917 accounts (86%) are Individual and 124 (14%) are
  Corporate.
- **Row level vs. account level:** Corporate accounts make up 14% of accounts but
  about 19% of rows. They lean toward full-season plans, so they have more event
  rows each. Always count accounts, not rows, when describing members.

## Renewal Dataset (889 accounts, one row per account)

### Messy values that need consolidating
- Renewal_Status has 10 spellings for 2 outcomes:
  - Renewed: "Renewed", "RENEWED", "renewed", "Y", "Yes"
  - Cancelled: "Cancelled", "CANCELLED", "Canceled", "N", "No"

### Early patterns
- **Cancellation_Reason:** 750 of 889 rows are blank, mostly because those members
  renewed. Of the 152 who cancelled, 139 gave a reason and 13 left it blank.
- **Top reasons**, roughly tied: Price increase (25), Bought elsewhere (25),
  Team performance (25), then Moving away (22) and Not attending enough (20).
- Cancellation_Reason is useful context for the business story, but it is
  **leakage**, so it can't be a model feature.

# Check for any Duplicates and Grain
##  Grain: What a single row represents 
### Ticketing Sheet: One row should pair one account with one event 

In [8]:
# Checking for Dublicate Rows within the Ticket Activity Data Set
tix.duplicated().sum() # Should show 0 

# Game Counts by Payment Plan in the Ticket Activity Data Set
tix["plan"] = tix["Price_Code"].str.strip().str.upper().str[-1] #Extracting the last Character of Price Code to determine ticket Plan (Full, Half, Quarter)
tix.groupby("Account_ID").agg(rows = ("Event_Name", "size"), plan =("plan","first")) \
   .groupby("plan")["rows"].describe() # Group by Account ID to gte the # of Rows per Acccount, then group by Plan to get the descriptive statitics for each plan type.



,count,mean,std,min,25%,50%,75%,max
plan,,,,,,,,
F,384.0,41.739583,2.016873,8.0,41.0,42.0,42.0,46.0
H,315.0,21.203175,1.157909,20.0,20.0,21.0,22.0,26.0
Q,218.0,10.518349,0.916884,10.0,10.0,10.0,11.0,13.0


## Findings: Duplicates and Grain
- **248 exact duplicate rows** (1%). Drop them before any counting.
- **Rows per account match each plan:**
  | Plan | Expected | Typical rows | Max (after dedup) |
  |---|---|---|---|
  | Full (F) | 41 games | 41–42 | 44 (41 + 3 concerts) |
  | Half (H) | 20–21 games | 20–22 | 24 |
  | Quarter (Q) | 10 games | 10–11 | 13 |
- Extra rows above the game count come from **concert add-ons** (0–3 per account).
  Duplicates also inflate the max until they're dropped (F: 46 → 44, H: 26 → 24).
- **One truncated account:** ACCT100916 has only 8 rows. It's the last account in
  the file, cut off when the export hit 25,000 rows. Its features (resale rate,
  no-show rate) would be based on 8 games, so needed to flag it, or drop it from modeling.
- Grain confirmed: one row = one account × one event.

# Join Check
## When Joining Tables for Modeling on Account_ID, need to make sure no ID's get silently dropped if the ID is not within the Renewal/Ticketing Dataset. 

In [9]:
# Join Check 

tix_ids = set(tix["Account_ID"])
ren_ids = set(ren["Account_ID"])

print("Match as-is:" , len(tix_ids & ren_ids))
print("Renewal IDs with no match:", len(ren_ids - tix_ids)) # Checks for IDs in renewal data that don't have a corresponding ID in the ticket data

#Uppercasing Renewal IDs to Match Ticket Activity IDs
ren_upper = set(ren["Account_ID"].str.upper())
print("Match after uppercasing:", len(tix_ids & ren_upper))
print("Renewal IDs with no match after upperccasing:", len(ren_upper - tix_ids)) # Checks for IDs in renewal data that don't have a corresponding ID in the ticket data after uppercasing the renewal IDs
print("Ticket Account IDs with no Match in Renewal Data:", len(tix_ids - ren_upper)) # 28

Match as-is: 875
Renewal IDs with no match: 14
Match after uppercasing: 889
Renewal IDs with no match after upperccasing: 0
Ticket Account IDs with no Match in Renewal Data: 28


## Join Check Summary
- **875 IDs match as-is.** The other 14 renewal IDs were lowercase versions of real
  IDs (e.g. "acct100457"). After uppercasing, **all 889 renewal IDs match**.
  Cleaning fix: `ren["Account_ID"] = ren["Account_ID"].str.upper()`
- **28 ticketing accounts have no renewal record** (917 − 889). With no outcome,
  they can't be used to train the model, so the modeling dataset is **889 accounts**.
  They can still be scored by the finished model to predict whether they'll renew.
- No renewal IDs are orphaned: every renewal record has ticket activity.

# Number Sanity Checks
## Purpose: Columns that should be numbers are stored as text. Before converting them, see what breaks and looks impossible. 


In [10]:
for col in ["Age", "Price_Per_Seat", "Num_Seats", "Tickets_Scanned", "Resale_Price"]:
    nums = pd.to_numeric(tix[col].str.replace("$", "", regex=False).str.strip(), errors="coerce") #strips dollar signs and whitespace, then converts to numberic values, coercing non-numberic values to NaN
    bad = nums.isna() & (tix[col].str.strip() != "") # blank or non-numeric values
    print(col, "| couldn't convert:", bad.sum())
    print(nums.describe().round(1), "\n")

Age | couldn't convert: 991
count    23047.0
mean        47.9
std         55.8
min         -1.0
25%         35.0
50%         45.0
75%         55.0
max        999.0
Name: Age, dtype: float64 

Price_Per_Seat | couldn't convert: 0
count    23730.0
mean       340.2
std        198.7
min         65.0
25%        137.2
50%        364.0
75%        528.0
max        747.5
Name: Price_Per_Seat, dtype: float64 

Num_Seats | couldn't convert: 0
count    24762.0
mean         3.3
std          1.8
min          1.0
25%          2.0
50%          3.0
75%          4.0
max          8.0
Name: Num_Seats, dtype: float64 

Tickets_Scanned | couldn't convert: 0
count    24501.0
mean         2.6
std          1.6
min          0.0
25%          1.0
50%          2.0
75%          4.0
max          8.0
Name: Tickets_Scanned, dtype: float64 

Resale_Price | couldn't convert: 0
count     505.0
mean      378.1
std       231.4
min        38.6
25%       165.4
50%       358.1
75%       545.6
max      1003.2
Name: Resale_Pric

In [11]:
# Logic Check --> Check if Number of Tickets Scanned is Greater than the # of Seats Purchased. If so, there is an error in the data.

(pd.to_numeric(tix["Tickets_Scanned"], errors = "coerce") > pd.to_numeric(tix["Num_Seats"], errors = "coerce")).sum() # Should be 0, if not, there is an error in the data

np.int64(0)

## Findings: Number Sanity Checks
| Column | Couldn't convert | Range | Action |
|---|---|---|---|
| Age | 991 | -1 to 999 | Text values ("34 yrs", "unknown") and impossible ages. Clean the text, then set ages under 18 or over 100 to NaN. |
| Price_Per_Seat | 0 (after removing "$") | $65–$748 | Clean. Convert to float. |
| Num_Seats | 0 | 1–8 | Clean. Convert to int. |
| Tickets_Scanned | 0 | 0–8 | Clean. Convert to int. |
| Resale_Price | 0 (after removing "$") | only filled on resold rows | Clean. Blank means not resold. |

- **Logic check:** Tickets_Scanned is never greater than Num_Seats (0 rows).
- **Age is the only messy numeric column.** Its mean (47.9) and std (55.8) are
  distorted by values like 999. The median (45) is the trustworthy number until
  it's cleaned.